# Proyecto BATSÚ — Notebook 03: Redacción del corpus con el modelo generador

**Universidad Cenfotec — MISIA** · Alvaro Granados, Jose Leonel Hernández

El notebook 02 produjo las transiciones estructuradas: para cada episodio se sabe qué acción tomó el agente en cada turno, cómo reaccionó el turista (estado afectivo, etapa AIDA) y cómo terminó. Lo que falta es el **texto de los mensajes**, que es lo único que el Juez RLAIF puede leer.

Este notebook lo redacta con el modelo generador abierto fijado en la Tabla 5.2, **Mistral-7B-Instruct v0.3** en cuantización de 4 bits, ejecutado en la GPU de Colab. El modelo no decide nada: recibe la traza estructurada del episodio ya resuelta por la tabla de reacción (decisión D3) y devuelve únicamente el diálogo en formato JSON. Es de familia distinta al juez (Llama 3) para evitar el sesgo de autopreferencia (decisión D4).

**Modo piloto.** La sección 2 tiene los parámetros `N_SEMILLAS` y `N_CORRIDAS`; por defecto redacta **una semilla, una corrida** para medir el tiempo por episodio y revisar la calidad del texto. La corrida completa se lanza cambiando esos dos valores a `None`. El guardado es incremental (una línea JSON por episodio), de modo que si la sesión de Colab se corta, al volver a correr solo se redactan los episodios que faltan.

**Secciones**
1. Configuración: Drive, GPU, librerías y acceso a Hugging Face
2. Parámetros de la corrida
3. Carga del corpus estructurado
4. Carga del modelo generador
5. Construcción del prompt a partir de la traza del episodio
6. Redacción por lotes con guardado incremental
7. Validación, muestra y manifiesto


## 1. Configuración

Se necesita una sesión de Colab **con GPU** (Entorno de ejecución → Cambiar tipo de entorno → T4 o L4). Se instalan `transformers`, `accelerate` y `bitsandbytes` (cuantización de 4 bits).

Mistral-7B-Instruct v0.3 está publicado en Hugging Face con acceso restringido: hay que aceptar sus condiciones en la página del modelo (una sola vez) y disponer de un token de lectura. El token se guarda en los *Secretos* de Colab (icono de llave en la barra lateral) con el nombre `HF_TOKEN`; así no queda escrito en el notebook.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import sys, json, time, re
import numpy as np
import pandas as pd

NOMBRE_CARPETA = 'DatosBatsuv2'
candidatos = [p for p in Path('/content/drive/MyDrive').rglob(NOMBRE_CARPETA) if p.is_dir()]
assert candidatos, f'No se encontró la carpeta "{NOMBRE_CARPETA}"'
BASE_DIR = candidatos[0]
CORPUS_DIR = BASE_DIR / 'corpus'
sys.path.insert(0, str(BASE_DIR))

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), 'Este notebook requiere GPU: cambiar el tipo de entorno de ejecución'
print('GPU:', torch.cuda.get_device_name(0))


Mounted at /content/drive
NVIDIA L4, 23034 MiB
GPU: NVIDIA L4


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes 2>&1 | tail -1

from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print('Sesión de Hugging Face iniciada')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 64.4 MB/s eta 0:00:00
Sesión de Hugging Face iniciada


## 2. Parámetros de la corrida

- **`ETIQUETA_CORPUS`**: qué corpus estructurado se redacta (el nombre que se le dio en el notebook 02).
- **`N_SEMILLAS`, `N_CORRIDAS`**: alcance del piloto. `1, 1` redacta un solo episodio; `None, None` redacta el corpus completo. En modo piloto se eligen episodios con al menos `MIN_TURNOS_PILOTO` turnos, para revisar un diálogo con varias interacciones y no uno que cierra en el primer mensaje.
- **`MODELO`**: identificador exacto en Hugging Face, que queda registrado en el manifiesto (decisión D4).
- **`IDIOMA_INTERNACIONAL`**: la Tabla 4.4 admite español neutro o inglés para el turista internacional. Se fija español neutro por defecto para que el juez evalúe todos los episodios en la misma lengua; cambiar a `'en'` si se quiere la variante bilingüe.
- **`TEMPERATURA`**: 0.7, el punto medio que la literatura de generación sintética recomienda entre repetición y deriva de formato.
- **`LOTE`**: episodios que se redactan en paralelo. 8 cabe en una T4; en una L4 se puede subir a 16.


In [ ]:
ETIQUETA_CORPUS = 'v0_escala05_prior40'
N_SEMILLAS = None            # None = todas
N_CORRIDAS = None            # None = todas
MIN_TURNOS_PILOTO = 3     # en modo piloto se eligen episodios de al menos esta longitud, para ver un diálogo real
MODELO = 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit'   # Mistral-7B-Instruct v0.3, pre-cuantizado a 4 bits (~4 GB)
IDIOMA_INTERNACIONAL = 'es'    # 'es' (español neutro) o 'en'
TEMPERATURA = 0.7
LOTE = 16
MAX_PALABRAS_MENSAJE = 40      # estilo WhatsApp (Sección 4.7)

RUTA_CORPUS = CORPUS_DIR / f'corpus_{ETIQUETA_CORPUS}.parquet'
RUTA_TEXTOS = CORPUS_DIR / f'textos_{ETIQUETA_CORPUS}.jsonl'      # una línea por episodio, guardado incremental
assert RUTA_CORPUS.exists(), f'No existe {RUTA_CORPUS}: correr el notebook 02 con esa etiqueta'
print('Corpus:', RUTA_CORPUS.name, '· textos:', RUTA_TEXTOS.name, '· modelo:', MODELO)


Corpus: corpus_v0_escala05_prior40.parquet · textos: textos_v0_escala05_prior40.jsonl · modelo: unsloth/mistral-7b-instruct-v0.3-bnb-4bit


## 3. Carga del corpus estructurado

Se carga el Parquet del notebook 02 junto con las semillas (para el contexto de cada episodio: origen, destino, tipo de turismo, presupuesto, activos). Se agrupa por episodio y se seleccionan los que corresponden al alcance del piloto. Los episodios que ya tengan texto en el archivo incremental se omiten.


In [1]:
import batsu_sim as bs
corpus = pd.read_parquet(RUTA_CORPUS)
semillas = bs.semillas.set_index('id')

episodios = corpus.sort_values(['id_semilla', 'corrida', 'turno']).groupby(['id_semilla', 'corrida'])
claves = list(episodios.groups.keys())
largo = episodios.size()
desenlace_ep = episodios['desenlace'].last()      # desenlace final de cada episodio

ids_sel = sorted({k[0] for k in claves})[:N_SEMILLAS] if N_SEMILLAS else sorted({k[0] for k in claves})
if N_CORRIDAS is None:
    claves = [k for k in claves if k[0] in ids_sel]
else:   # piloto: las primeras N_CORRIDAS corridas de cada semilla con al menos MIN_TURNOS_PILOTO turnos
    claves = [k for sid in ids_sel for k in [c for c in claves if c[0] == sid and largo[c] >= MIN_TURNOS_PILOTO][:N_CORRIDAS]]

def validar_turnos(turnos, n, desenlace):
    """Valida el esquema del diálogo (no solo que sea JSON): n turnos, cada uno con 'agente' y 'turista'
    de tipo texto, no vacíos (salvo el último mensaje del turista en un abandono) y sin caracteres dañados."""
    if not isinstance(turnos, list) or len(turnos) != n:
        return False, 'numero_de_turnos'
    for i, t in enumerate(turnos):
        if not isinstance(t, dict):
            return False, f'turno_{i+1}_no_es_objeto'
        for campo in ('agente', 'turista'):
            v = t.get(campo)
            if not isinstance(v, str):
                return False, f'turno_{i+1}_{campo}_tipo'
            if '\ufffd' in v:
                return False, f'turno_{i+1}_{campo}_caracteres_danados'
            vacio_permitido = campo == 'turista' and i == n - 1 and desenlace == 'abandono'
            if not v.strip() and not vacio_permitido:
                return False, f'turno_{i+1}_{campo}_vacio'
    return True, None

# Lo ya redactado solo cuenta si pasa la validación de esquema; el último registro de cada episodio prevalece
ultimo = {}
if RUTA_TEXTOS.exists():
    for linea in open(RUTA_TEXTOS, encoding='utf-8'):
        d = json.loads(linea)
        ultimo[(d['id_semilla'], d['corrida'])] = d
hechos, a_rehacer = set(), 0
for k, d in ultimo.items():
    if k in largo.index and d.get('valido') and validar_turnos(d.get('turnos'), int(largo[k]), desenlace_ep[k])[0]:
        hechos.add(k)
    else:
        a_rehacer += 1

pendientes = sorted([k for k in claves if k not in hechos], key=lambda k: largo[k])   # lotes homogéneos en longitud
print(f'Episodios en alcance: {len(claves)} · ya redactados y válidos: {len(hechos)} · '
      f'a rehacer por esquema: {a_rehacer} · pendientes: {len(pendientes)}')


ModuleNotFoundError: No module named 'batsu_sim'

## 4. Carga del modelo generador

Se carga Mistral-7B-Instruct v0.3 con cuantización de 4 bits (NF4, cómputo en float16), que ocupa unos 4.5 GB de VRAM y cabe en la T4 gratuita. El tokenizador se configura con relleno a la izquierda, necesario para generar varios episodios en un mismo lote.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODELO, token=HF_TOKEN)
tok.pad_token = tok.eos_token
tok.padding_side = 'left'
modelo = AutoModelForCausalLM.from_pretrained(MODELO, device_map='auto', token=HF_TOKEN)
modelo.eval()
print(f'Modelo cargado en {time.time()-t0:.0f} s · VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB')


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.14GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

Modelo cargado en 29 s · VRAM usada: 4.1 GB


## 5. Construcción del prompt a partir de la traza del episodio

El prompt sigue la estructura de rol, contexto y restricciones de salida de la Sección 4.7. Contiene:

- **Rol**: redactor de un diálogo de WhatsApp entre el turista y la MiPyme.
- **Contexto de la semilla**: segmento, origen y destino, tipo de turismo, presupuesto, activos disponibles, estado afectivo inicial con sus marcadores lingüísticos (Tabla 4.2).
- **Traza turno a turno**, ya decidida por la tabla de reacción: la acción del agente (con el activo enviado si aplica), el estado afectivo del turista antes y después, si la acción violó el Trust Gate y, en el último turno, el desenlace.
- **Restricciones**: un mensaje del agente y una respuesta del turista por turno, máximo 40 palabras cada uno, estilo WhatsApp, y salida **únicamente** en JSON con la lista de turnos.

Para el turista nacional se pide español costarricense coloquial; para el internacional, español neutro (o inglés, según el parámetro). Los estados afectivos se describen con los marcadores de la Tabla 4.2 para que el texto los refleje de forma consistente, que es lo que después el juez debe reconocer.


In [ ]:
MARCADORES = {   # Tabla 4.2
    'entusiasmado': 'signos de exclamación, emojis, superlativos ("me encanta", "buenísimo")',
    'indeciso': 'preguntas comparativas, condicionales ("no sé si...", "tal vez")',
    'ansioso': 'preguntas repetidas sobre tiempo o seguridad, urgencia ("necesito confirmar ya")',
    'desconfiado': 'preguntas de verificación ("¿es oficial?", "¿tienen reseñas?"), tono cauteloso',
    'frustrado': 'quejas explícitas, lenguaje cortante, referencias a intentos previos fallidos',
}
DESCRIPCION_ACCION = {   # Tabla 5.8
    'solicitar_informacion': 'el agente pide un dato que le falta (fechas, presupuesto, número de personas)',
    'ofrecer_paquete_bajo': 'el agente ofrece un paquete económico',
    'ofrecer_paquete_medio': 'el agente ofrece un paquete de precio medio',
    'ofrecer_paquete_alto': 'el agente ofrece un paquete de precio alto',
    'enviar_activo': 'el agente envía un activo promocional',
    'escalar_humano': 'el agente transfiere la conversación a un asesor humano, que responde en ese mismo turno',
    'cerrar_conversacion': 'el agente cierra la conversación cordialmente sin nueva oferta',
}
DESENLACE_TXT = {
    'conversion': 'el turista confirma la reserva',
    'avance_parcial': 'el turista se despide interesado pero sin reservar todavía',
    'rechazo': 'el turista declina explícitamente la oferta',
    'abandono': 'el turista deja de responder (su último mensaje queda vacío o es una despedida evasiva)',
    'recomendacion_inviable': 'el turista señala que lo ofrecido es imposible para él por la distancia o el traslado, y se molesta',
    'finalizacion_limite_turnos': 'la conversación se agota sin cierre; el turista se despide sin decidir',
}

def construir_prompt(id_sem, corrida):
    s = semillas.loc[id_sem]
    tr = episodios.get_group((id_sem, corrida)).sort_values('turno')
    nacional = s['segmento'] == 'nacional'
    if nacional:
        lengua = 'español costarricense coloquial (voseo, expresiones locales como "mae", "pura vida", "diay")'
    else:
        lengua = 'español neutro, claro y sin modismos' if IDIOMA_INTERNACIONAL == 'es' else 'English (the tourist writes in English; the business answers in English)'
    activos = ', '.join(s['activos_lista'])
    lineas = []
    for r in tr.itertuples():
        det = DESCRIPCION_ACCION[r.accion_nombre]
        if r.accion_nombre == 'enviar_activo' and r.activo_enviado:
            det += f' ({r.activo_enviado})'
        if r.enmascarada:
            det = 'el agente intenta enviar un activo que no tiene; solo puede responder con texto'
        nota = ''
        if r.violacion_trust_gate and 'M2' in (r.modificadores or ''):
            nota = ' [la oferta supera el presupuesto del turista]'
        if r.violacion_trust_gate and 'M1' in (r.modificadores or ''):
            nota = ' [la oferta es geográficamente inviable para el turista]'
        fin = f' → FIN: {DESENLACE_TXT[r.desenlace]}' if r.desenlace else ''
        cambio = f'sigue {r.estado_despues}' if r.estado_antes == r.estado_despues else f'pasa de {r.estado_antes} a {r.estado_despues}'
        lineas.append(f'Turno {r.turno}: {det}{nota}. El turista {cambio}{fin}.')
    traza = '\n'.join(lineas)
    return f"""Actúe como redactor de un diálogo de WhatsApp entre un turista y una MiPyme turística de Costa Rica.
El diálogo ya está decidido turno a turno; su única tarea es redactar los mensajes con naturalidad, en {lengua}.

CONTEXTO DEL CASO
- Turista {s['segmento']}, viaja desde {s['origen']} hacia {s['destino']} ({s['tipo_turismo']}), presupuesto {s['presupuesto']}.
- Activos que la MiPyme tiene disponibles: {activos}.
- Estado afectivo inicial del turista: {s['estado_afectivo_inicial']} ({MARCADORES[s['estado_afectivo_inicial']]}).
- Cuando el estado afectivo cambia, el mensaje del turista debe reflejarlo con sus marcadores: {'; '.join(f'{k}: {v}' for k, v in MARCADORES.items())}.

TRAZA DEL EPISODIO (no la altere)
{traza}

RESTRICCIONES DE SALIDA
- Exactamente {len(tr)} turnos. En cada turno, primero escribe el agente y luego responde el turista.
- Máximo {MAX_PALABRAS_MENSAJE} palabras por mensaje, estilo WhatsApp (breve, directo, puede usar emojis con moderación).
- No invente precios exactos ni nombres de personas; puede decir "el paquete económico", "unos 45 mil colones" es aceptable solo si es coherente con el presupuesto.
- Responda ÚNICAMENTE con JSON válido, sin texto antes ni después, con esta forma:
{{"turnos": [{{"turno": 1, "agente": "...", "turista": "..."}}, ...]}}"""

print(construir_prompt(*pendientes[0]) if pendientes else construir_prompt(*claves[0]))


Actúe como redactor de un diálogo de WhatsApp entre un turista y una MiPyme turística de Costa Rica.
El diálogo ya está decidido turno a turno; su única tarea es redactar los mensajes con naturalidad, en español costarricense coloquial (voseo, expresiones locales como "mae", "pura vida", "diay").

CONTEXTO DEL CASO
- Turista nacional, viaja desde San Jose hacia Tamarindo (playa), presupuesto bajo.
- Activos que la MiPyme tiene disponibles: texto.
- Estado afectivo inicial del turista: entusiasmado (signos de exclamación, emojis, superlativos ("me encanta", "buenísimo")).
- Cuando el estado afectivo cambia, el mensaje del turista debe reflejarlo con sus marcadores: entusiasmado: signos de exclamación, emojis, superlativos ("me encanta", "buenísimo"); indeciso: preguntas comparativas, condicionales ("no sé si...", "tal vez"); ansioso: preguntas repetidas sobre tiempo o seguridad, urgencia ("necesito confirmar ya"); desconfiado: preguntas de verificación ("¿es oficial?", "¿tienen reseñas?

## 6. Redacción por lotes con guardado incremental

Cada lote de episodios se convierte en prompts con la plantilla de chat de Mistral, se genera con muestreo (temperatura 0.7) y se extrae el JSON de la respuesta. Si la respuesta no es JSON válido o no tiene el número de turnos esperado, el episodio se marca con `valido: false` y se reintenta una vez; si vuelve a fallar, queda registrado para revisión y no bloquea el resto.

Cada episodio redactado se escribe de inmediato como una línea en `textos_<etiqueta>.jsonl`. Al terminar cada lote se imprime el tiempo acumulado y una **proyección del tiempo total** para el corpus completo, que es el dato que se necesita para dimensionar la corrida definitiva.


In [ ]:
def extraer_json(texto):
    m = re.search(r'\{.*\}', texto, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def generar_lote(prompts, max_turnos):
    mensajes = [[{'role': 'user', 'content': p}] for p in prompts]
    textos = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in mensajes]
    entradas = tok(textos, return_tensors='pt', padding=True).to(modelo.device)
    max_new = min(1800, 140 * max_turnos + 80)
    with torch.no_grad():
        salida = modelo.generate(**entradas, max_new_tokens=max_new, do_sample=True, temperature=TEMPERATURA, top_p=0.9,
                                 pad_token_id=tok.eos_token_id)
    nuevos = salida[:, entradas['input_ids'].shape[1]:]
    return tok.batch_decode(nuevos, skip_special_tokens=True)

def redactar(claves_lote, intento=1):
    prompts = [construir_prompt(*k) for k in claves_lote]
    n_turnos = [len(episodios.get_group(k)) for k in claves_lote]
    respuestas = generar_lote(prompts, max(n_turnos))
    resultados, fallidos = [], []
    for k, n, resp in zip(claves_lote, n_turnos, respuestas):
        d = extraer_json(resp)
        ok = d is not None and isinstance(d.get('turnos'), list) and len(d['turnos']) == n
        if ok:
            resultados.append({'id_semilla': k[0], 'corrida': k[1], 'valido': True, 'turnos': d['turnos'], 'modelo': MODELO, 'intento': intento})
        elif intento == 1:
            fallidos.append(k)
        else:
            resultados.append({'id_semilla': k[0], 'corrida': k[1], 'valido': False, 'turnos': None, 'crudo': resp[:2000], 'modelo': MODELO, 'intento': intento})
    if fallidos:
        resultados += redactar(fallidos, intento=2)
    return resultados

total = len(pendientes); hechos_ahora = 0; t0 = time.time()
with open(RUTA_TEXTOS, 'a', encoding='utf-8') as f:
    for i in range(0, total, LOTE):
        lote = pendientes[i:i + LOTE]
        for r in redactar(lote):
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
        f.flush()
        hechos_ahora += len(lote)
        seg = time.time() - t0
        por_ep = seg / hechos_ahora
        n_corpus = corpus.groupby(['id_semilla', 'corrida']).ngroups
        print(f'{hechos_ahora}/{total} episodios · {seg/60:.1f} min · {por_ep:.1f} s/episodio · '
              f'proyección corpus completo ({n_corpus} ep.): {por_ep*n_corpus/3600:.1f} h')
print('Redacción terminada' if total else 'No había episodios pendientes')


No había episodios pendientes


## 7. Validación, muestra y manifiesto

Se relee el archivo incremental, se cuentan los episodios válidos e inválidos, se verifica la longitud de los mensajes contra el límite de 40 palabras y se muestra un episodio completo para revisar la calidad: que el estado afectivo se note en el texto, que la acción del agente coincida con la traza y que el cierre corresponda al desenlace.

Finalmente se actualiza el manifiesto del corpus con el modelo, la versión, los parámetros de generación y los conteos.


In [ ]:
registros = [json.loads(l) for l in open(RUTA_TEXTOS, encoding='utf-8')]
validos = [r for r in registros if r['valido']]
print(f'Episodios redactados: {len(registros)} · válidos: {len(validos)} · inválidos: {len(registros)-len(validos)}')

palabras = [len(t[c].split()) for r in validos for t in r['turnos'] for c in ('agente', 'turista') if t.get(c)]
if palabras:
    print(f'Palabras por mensaje: media {np.mean(palabras):.1f} · máx {max(palabras)} · > {MAX_PALABRAS_MENSAJE}: {np.mean(np.array(palabras) > MAX_PALABRAS_MENSAJE):.1%}')

# Muestra: último episodio válido, con su traza al lado
if validos:
    r = validos[-1]; k = (r['id_semilla'], r['corrida']); tr = episodios.get_group(k).sort_values('turno')
    s = semillas.loc[k[0]]
    print(f'\n=== {k[0]} corrida {k[1]} · {s["segmento"]} · {s["tipo_turismo"]} · {s["origen"]} → {s["destino"]} · esperado: {s["resultado_esperado"]} ===')
    for t, row in zip(r['turnos'], tr.itertuples()):
        print(f'[{row.turno}] ({row.accion_nombre}; {row.estado_antes}→{row.estado_despues}{" · " + row.desenlace if row.desenlace else ""})')
        print(f'    MiPyme:  {t.get("agente")}')
        print(f'    Turista: {t.get("turista")}')

# Manifiesto
ruta_manif = CORPUS_DIR / f'manifiesto_{ETIQUETA_CORPUS}.json'
manif = json.load(open(ruta_manif, encoding='utf-8'))
manif['generador'] = {'modelo': MODELO, 'cuantizacion': '4bit-nf4', 'temperatura': TEMPERATURA, 'top_p': 0.9,
                      'idioma_internacional': IDIOMA_INTERNACIONAL, 'max_palabras_mensaje': MAX_PALABRAS_MENSAJE,
                      'transformers': __import__('transformers').__version__, 'fecha': time.strftime('%Y-%m-%d %H:%M:%S'),
                      'episodios_redactados': len(registros), 'episodios_validos': len(validos),
                      'completo': len(validos) == corpus.groupby(['id_semilla', 'corrida']).ngroups}
json.dump(manif, open(ruta_manif, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('\nManifiesto actualizado:', ruta_manif.name)


Episodios redactados: 128 · válidos: 127 · inválidos: 1
Palabras por mensaje: media 15.2 · máx 52 · > 40: 0.7%

=== CR-TUR-001 corrida 21 · nacional · playa · San Jose → Tamarindo · esperado: conversion ===
[1] (ofrecer_paquete_bajo; entusiasmado→entusiasmado)
    MiPyme:  Hola! 🌴 Estoy de la empresa Mipyme Turística, tenemos un paquete económico para ti desde San José hasta Tamarindo 🏖️. ¿Te interesa?
    Turista: Sí, pura vida! Me encanta la idea 🤩
[2] (ofrecer_paquete_bajo; entusiasmado→entusiasmado)
    MiPyme:  Me alegra que te guste. El paquete incluye transporte, alojamiento básico y desayuno 🍳. ¿Te parece bien?
    Turista: Buenísimo, gracias 👍
[3] (ofrecer_paquete_bajo; entusiasmado→entusiasmado)
    MiPyme:  Además, puedes disfrutar de actividades como surf, kayak y paseos a caballo en Tamarindo 🏄‍♂️🚣‍♂️🐎. ¿Tal vez te gustaría hacer alguna de ellas?
    Turista: ¡Sí, me encantaría hacer surf! 🌊
[4] (ofrecer_paquete_bajo; entusiasmado→entusiasmado)
    MiPyme:  Genial! El paqu